In [ ]:
from bigmodule import M, I

# <aistudiograph>

# @param(id="m7", name="initialize")
# 交易引擎：初始化函数，只执行一次
def m7_initialize_bigquant_run(context):
    from bigtrader.finance.commission import PerOrder

    # 系统已经设置了默认的交易手续费和滑点，要修改手续费可使用如下函数
    context.set_commission(PerOrder(buy_cost=0.0003, sell_cost=0.0013, min_cost=5))
# @param(id="m7", name="before_trading_start")
# 交易引擎：每个单位时间开盘前调用一次。
def m7_before_trading_start_bigquant_run(context, data):
    # 盘前处理，订阅行情等
    pass

# @param(id="m7", name="handle_tick")
# 交易引擎：tick数据处理函数，每个tick执行一次
def m7_handle_tick_bigquant_run(context, tick):
    pass

# @param(id="m7", name="handle_data")
def m7_handle_data_bigquant_run(context, data):
    import pandas as pd

    # 下一个交易日不是调仓日，则不生成信号
    if not context.rebalance_period.is_signal_date(data.current_dt.date()):
        return

    # 从传入的数据 context.data 中读取今天的信号数据
    today_df = context.data[context.data["date"] == data.current_dt.strftime("%Y-%m-%d")]

    # 卖出不在目标持有列表中的股票
    for instrument in sorted(set(context.get_account_positions().keys()) - set(today_df["instrument"])):
        context.order_target_percent(instrument, 0)
    # 买入目标持有列表中的股票
    for i, x in today_df.iterrows():
        context.order_target_percent(x.instrument, 0.0 if pd.isnull(x.position) else x.position)
# @param(id="m7", name="handle_trade")
# 交易引擎：成交回报处理函数，每个成交发生时执行一次
def m7_handle_trade_bigquant_run(context, trade):
    pass

# @param(id="m7", name="handle_order")
# 交易引擎：委托回报处理函数，每个委托变化时执行一次
def m7_handle_order_bigquant_run(context, order):
    pass

# @param(id="m7", name="after_trading")
# 交易引擎：盘后处理函数，每日盘后执行一次
def m7_after_trading_bigquant_run(context, data):
    pass

# @module(position="-85,-169", comment="""因子特征，用表达式构建因子""")
m1 = M.input_features_dai.v30(
    mode="""表达式""",
    expr="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home
-- 数据使用: 表名.字段名, 对于没有指定表名的列, 会从 expr_tables 推断, 如果同名字段在多个表中出现, 需要显式的给出表名

-- cn_stock_prefactors https://bigquant.com/data/datas

-- 价值/估值
pe_ttm
pb
ps_ttm
pcf_net_leading
pcf_op_ttm
total_market_cap
float_market_cap
dividend_yield_ratio
-- 5日动量/反转
momentum_5
reversal_5
-- 5日波动
volatility_5
-- 换手
turn

-- cn_stock_bar1d.close / cn_stock_bar1d.openources/cn_stock_prefactors 是常用因子表(VIEW), JOIN了很多数据表, 性能会比直接用相关表慢一点, 但使用简单
-- cn_stock_prefactors.pe_ttm

-- 表达式模式下, 会自动join输入数据1/2/3, 可以在表达式里直接使用其字段。包括 input_1 的所有列但去掉 date, instrument。注意字段不能有重复的, 否则会报错
-- input_1.* EXCLUDE(date, instrument)
-- input_1.close
-- input_2.close / input_1.close
""",
    expr_filters="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home
-- 表达式模式的过滤都是放在 QUALIFY 里, 即数据查询、计算, 最后才到过滤条件
st_status = 0
list_days > 260
-- c_pct_rank(-return_90) <= 0.3
-- c_pct_rank(return_30) <= 0.3
-- cn_stock_bar1d.turn > 0.02
""",
    expr_tables="""cn_stock_prefactors""",
    extra_fields="""date, instrument""",
    order_by="""date, instrument""",
    expr_drop_na=True,
    sql="""-- 使用DAI SQL获取数据, 构建因子等, 如下是一个例子作为参考
-- DAI SQL 语法: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-sql%E5%85%A5%E9%97%A8%E6%95%99%E7%A8%8B
-- 使用数据输入1/2/3里的字段: e.g. input_1.close, input_1.* EXCLUDE(date, instrument)

SELECT
    -- 在这里输入因子表达式
    -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
    -- 数据&字段: 数据文档 https://bigquant.com/data/home

    m_lag(close, 90) / close AS return_90,
    m_lag(close, 30) / close AS return_30,
    -- 下划线开始命名的列是中间变量, 不会在最终结果输出 (e.g. _rank_return_90)
    c_pct_rank(-return_90) AS _rank_return_90,
    c_pct_rank(return_30) AS _rank_return_30,

    c_rank(volume) AS rank_volume,
    close / m_lag(close, 1) as return_0,

    -- 日期和股票代码
    date, instrument
FROM
    -- 预计算因子 cn_stock_bar1d https://bigquant.com/data/datasources/cn_stock_bar1d
    cn_stock_prefactors
    -- SQL 模式不会自动join输入数据源, 可以根据需要自由灵活的使用
    -- JOIN input_1 USING(date, instrument)
WHERE
    -- WHERE 过滤, 在窗口等计算算子之前执行
    -- 剔除ST股票
    st_status = 0
QUALIFY
    -- QUALIFY 过滤, 在窗口等计算算子之后执行, 比如 m_lag(close, 3) AS close_3, 对于 close_3 的过滤需要放到这里
    -- 去掉有空值的行
    COLUMNS(*) IS NOT NULL
    -- _rank_return_90 是窗口函数结果，需要放在 QUALIFY 里
    AND _rank_return_90 > 0.1
    AND _rank_return_30 < 0.1
-- 按日期和股票代码排序, 从小到大
ORDER BY date, instrument
""",
    extract_data=False,
    m_name="""m1"""
)

# @module(position="-258,-25", comment="""加数据标注""")
m2 = M.input_features_dai.v30(
    input_1=m1.data,
    mode="""表达式""",
    expr="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home
-- 数据使用: 表名.字段名, 对于没有指定表名的列, 会从 expr_tables 推断, 如果同名字段在多个表中出现, 需要显式的给出表名

input_1.* EXCLUDE(date, instrument)
m_lead(close, 5) / m_lead(open, 1) AS _future_return
c_quantile_cont(_future_return, 0.01) AS _future_return_1pct
c_quantile_cont(_future_return, 0.99) AS _future_return_99pct
clip(_future_return, _future_return_1pct, _future_return_99pct) AS _clipped_return
c_cbins(_clipped_return, 20) AS label

-- cn_stock_bar1d.close / cn_stock_bar1d.open
-- cn_stock_prefactors https://bigquant.com/data/datasources/cn_stock_prefactors 是常用因子表(VIEW), JOIN了很多数据表, 性能会比直接用相关表慢一点, 但使用简单
-- cn_stock_prefactors.pe_ttm

-- 表达式模式下, 会自动join输入数据1/2/3, 可以在表达式里直接使用其字段。包括 input_1 的所有列但去掉 date, instrument。注意字段不能有重复的, 否则会报错
-- input_1.* EXCLUDE(date, instrument)
-- input_1.close
-- input_2.close / input_1.close
""",
    expr_filters="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 数据&字段: 数据文档 https://bigquant.com/data/home
-- 表达式模式的过滤都是放在 QUALIFY 里, 即数据查询、计算, 最后才到过滤条件

-- c_pct_rank(-return_90) <= 0.3
-- c_pct_rank(return_30) <= 0.3
-- cn_stock_bar1d.turn > 0.02
""",
    expr_tables="""cn_stock_prefactors""",
    extra_fields="""date, instrument""",
    order_by="""date, instrument""",
    expr_drop_na=True,
    sql="""-- 使用DAI SQL获取数据, 构建因子等, 如下是一个例子作为参考
-- DAI SQL 语法: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-sql%E5%85%A5%E9%97%A8%E6%95%99%E7%A8%8B
-- 使用数据输入1/2/3里的字段: e.g. input_1.close, input_1.* EXCLUDE(date, instrument)

SELECT
    -- 在这里输入因子表达式
    -- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
    -- 数据&字段: 数据文档 https://bigquant.com/data/home

    m_lag(close, 90) / close AS return_90,
    m_lag(close, 30) / close AS return_30,
    -- 下划线开始命名的列是中间变量, 不会在最终结果输出 (e.g. _rank_return_90)
    c_pct_rank(-return_90) AS _rank_return_90,
    c_pct_rank(return_30) AS _rank_return_30,

    c_rank(volume) AS rank_volume,
    close / m_lag(close, 1) as return_0,

    -- 日期和股票代码
    date, instrument
FROM
    -- 预计算因子 cn_stock_bar1d https://bigquant.com/data/datasources/cn_stock_bar1d
    cn_stock_prefactors
    -- SQL 模式不会自动join输入数据源, 可以根据需要自由灵活的使用
    -- JOIN input_1 USING(date, instrument)
WHERE
    -- WHERE 过滤, 在窗口等计算算子之前执行
    -- 剔除ST股票
    st_status = 0
QUALIFY
    -- QUALIFY 过滤, 在窗口等计算算子之后执行, 比如 m_lag(close, 3) AS close_3, 对于 close_3 的过滤需要放到这里
    -- 去掉有空值的行
    COLUMNS(*) IS NOT NULL
    -- _rank_return_90 是窗口函数结果，需要放在 QUALIFY 里
    AND _rank_return_90 > 0.1
    AND _rank_return_30 < 0.1
-- 按日期和股票代码排序, 从小到大
ORDER BY date, instrument
""",
    extract_data=False,
    m_name="""m2"""
)

# @module(position="-260,78", comment="""抽取训练数据""")
m3 = M.extract_data_dai.v20(
    sql=m2.data,
    start_date="""2021-01-01""",
    start_date_bound_to_trading_date=False,
    end_date="""2022-12-31""",
    end_date_bound_to_trading_date=False,
    before_start_days=90,
    keep_before=False,
    debug=False,
    m_name="""m3"""
)

# @module(position="102,-22", comment="""抽取预测数据""")
m4 = M.extract_data_dai.v20(
    sql=m1.data,
    start_date="""2023-01-01""",
    start_date_bound_to_trading_date=True,
    end_date="""2024-01-01""",
    end_date_bound_to_trading_date=True,
    before_start_days=90,
    keep_before=False,
    debug=False,
    m_name="""m4"""
)

# @module(position="-126,211", comment="""模型训练""")
m5 = M.stockranker.v9(
    train_data=m3.data,
    predict_data=m4.data,
    learning_algorithm="""排序""",
    number_of_leaves=30,
    min_docs_per_leaf=1000,
    number_of_trees=20,
    learning_rate=0.1,
    max_bins=1023,
    feature_fraction=1,
    data_row_fraction=1,
    sort_by="""date,instrument""",
    plot_charts=True,
    ndcg_discount_base=1,
    m_name="""m5"""
)

# @module(position="-74,302", comment="""等权分配""")
m6 = M.score_to_position.v4(
    input_1=m5.predictions,
    score_field="""score DESC""",
    hold_count=10,
    position_expr="""-- DAI SQL 算子/函数: https://bigquant.com/wiki/doc/dai-PLSbc1SbZX#h-%E5%87%BD%E6%95%B0
-- 在这里输入表达式, 每行一个表达式, 输出仓位字段必须命名为 position, 模块会进一步做归一化
-- 排序倒数: 1 / score_rank AS position
-- 对数下降: 1 / log2(score_rank + 1) AS position
-- TODO 拟合、最优化 ..

-- 等权重分配
1 AS position
""",
    total_position=1,
    extract_data=True,
    m_name="""m6"""
)

# @module(position="-106,405", comment="""交易，日线，设置初始化函数和K线处理函数，以及初始化资金、基准等""")
m7 = M.bigtrader.v47(
    data=m6.data,
    start_date="""""",
    end_date="""""",
    initialize=m7_initialize_bigquant_run,
    before_trading_start=m7_before_trading_start_bigquant_run,
    handle_tick=m7_handle_tick_bigquant_run,
    handle_data=m7_handle_data_bigquant_run,
    handle_trade=m7_handle_trade_bigquant_run,
    handle_order=m7_handle_order_bigquant_run,
    after_trading=m7_after_trading_bigquant_run,
    capital_base=1000000,
    frequency="""daily""",
    product_type="""股票""",
    rebalance_period_type="""交易日""",
    rebalance_period_days="""5""",
    rebalance_period_roll_forward=True,
    backtest_engine_mode="""标准模式""",
    before_start_days=0,
    volume_limit=1,
    order_price_field_buy="""open""",
    order_price_field_sell="""open""",
    benchmark="""沪深300指数""",
    plot_charts=True,
    debug=False,
    backtest_only=False,
    m_name="""m7"""
)
# </aistudiograph>

[2025-09-02 15:00:32] [info     ] input_features_dai.v30 开始运行 ..
[2025-09-02 15:00:35] [info     ] input_features_dai.v30 命中缓存
[2025-09-02 15:00:35] [info     ] input_features_dai.v30 运行完成 [3.096s].
[2025-09-02 15:00:35] [info     ] input_features_dai.v30 开始运行 ..
[2025-09-02 15:00:35] [info     ] input_features_dai.v30 命中缓存
[2025-09-02 15:00:35] [info     ] input_features_dai.v30 运行完成 [0.112s].
[2025-09-02 15:00:35] [info     ] extract_data_dai.v20 开始运行 ..
[2025-09-02 15:00:35] [info     ] extract_data_dai.v20 命中缓存
[2025-09-02 15:00:35] [info     ] extract_data_dai.v20 运行完成 [0.118s].
[2025-09-02 15:00:35] [info     ] extract_data_dai.v20 开始运行 ..
[2025-09-02 15:00:35] [info     ] extract_data_dai.v20 命中缓存
[2025-09-02 15:00:35] [info     ] extract_data_dai.v20 运行完成 [0.084s].
[2025-09-02 15:00:35] [info     ] stockranker.v9 开始运行 ..
[2025-09-02 15:00:35] [info     ] stockranker.v9 命中缓存


[2025-09-02 15:00:36] [info     ] stockranker.v9 运行完成 [0.772s].
[2025-09-02 15:00:36] [info     ] score_to_position.v4 开始运行 ..
[2025-09-02 15:00:36] [info     ] score_to_position.v4 命中缓存
[2025-09-02 15:00:36] [info     ] score_to_position.v4 运行完成 [0.103s].
[2025-09-02 15:00:38] [info     ] bigtrader.v47 开始运行 ..
[2025-09-02 15:00:38] [info     ] bigtrader.v47 命中缓存


[2025-09-02 15:00:42] [info     ] bigtrader.v47 运行完成 [3.822s].
